# JurisGuard V2 — Phi-3.5 Legal Fine-tune (Colab)

**Crash-safe training:** every checkpoint saves **model + optimizer + scheduler + RNG** to Google Drive.
If Colab disconnects for hours or days, re-run all cells — training auto-resumes from `checkpoint_RESUME/`.

## Before you start
1. Upload to Drive: `My Drive/JurisGuard/training/train_final.jsonl` and `eval_set.jsonl`
2. Runtime → **Change runtime type → T4 GPU**
3. Run local smoke test first: `python scripts/05_smoke_test_finetune.py`

## If Unsloth / bitsandbytes errors
**Runtime → Restart session** → run cells **1 → 3** → then continue from cell 4.

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Config — edit paths if needed
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/JurisGuard/training')
TRAIN_JSONL = DRIVE_ROOT / 'train_final.jsonl'
EVAL_JSONL = DRIVE_ROOT / 'eval_set.jsonl'

OUTPUT_DIR = DRIVE_ROOT / 'checkpoints'
RESUME_DIR = DRIVE_ROOT / 'checkpoint_RESUME'   # always-latest full state
TOKENIZED_CACHE = DRIVE_ROOT / 'tokenized_cache'
MANIFEST_PATH = DRIVE_ROOT / 'RUN_MANIFEST.json'
FINAL_ADAPTER = DRIVE_ROOT / 'final_adapter'
GGUF_OUTPUT = DRIVE_ROOT / 'gguf'

MODEL_ID = 'unsloth/Phi-3.5-mini-instruct'
MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 1
SAVE_STEPS = 200          # lose at most 200 steps on crash
SAVE_TOTAL_LIMIT = 5      # keep last 5 rolling checkpoints
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4

for p in [OUTPUT_DIR, RESUME_DIR, TOKENIZED_CACHE, FINAL_ADAPTER, GGUF_OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

assert TRAIN_JSONL.is_file(), f'Upload train_final.jsonl to {TRAIN_JSONL}'
print('Drive paths OK')
print('  train:', TRAIN_JSONL)
print('  resume:', RESUME_DIR)

In [ ]:
# @title 3. Install Unsloth (Colab-safe stack) — then continue to cell 4
import os, re, sys, torch

# Colab-safe pinned install (avoids broken bitsandbytes.functional)
v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")

get_ipython().system("pip uninstall -y unsloth unsloth_zoo bitsandbytes trl peft 2>/dev/null | tail -3")
get_ipython().system(f"pip install -q --no-deps bitsandbytes accelerate {xformers} peft 'trl<0.23.0' triton cut_cross_entropy unsloth_zoo")
get_ipython().system("pip install -q sentencepiece protobuf 'datasets>=3.4.1,<4.0.0' 'huggingface_hub>=0.34.0' hf_transfer")
get_ipython().system("pip install -q --no-deps 'unsloth @ git+https://github.com/unslothai/unsloth.git'")

# Clear half-loaded modules from earlier failed imports
for name in list(sys.modules):
    if any(x in name for x in ("unsloth", "bitsandbytes", "peft", "trl")):
        del sys.modules[name]

import unsloth  # MUST be before transformers
from unsloth import FastLanguageModel
import bitsandbytes as bnb

assert hasattr(bnb, "functional"), (
    "bitsandbytes still broken. Do: Runtime → Restart session → re-run cells 1-3 only, then cell 4+"
)
print("Unsloth OK")
print("  torch:", torch.__version__)
print("  bitsandbytes:", bnb.__version__)
print("  GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# @title 4. Resume helpers (full optimizer state)
import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

from transformers import TrainerCallback

SYSTEM_PROMPT = 'You are JurisGuard, an expert legal contract analyst.'


def format_example(example, tokenizer):
    instruction = (example.get('instruction') or '').strip()
    user_input = (example.get('input') or '').strip()
    output = (example.get('output') or '').strip()
    user_content = f'{instruction}\n\n{user_input}'.strip() if user_input else instruction
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': output},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def find_latest_checkpoint(output_dir: Path):
    if not output_dir.is_dir():
        return None
    best, best_step = None, -1
    for path in output_dir.iterdir():
        m = re.fullmatch(r'checkpoint-(\d+)', path.name)
        if not m or not (path / 'trainer_state.json').is_file():
            continue
        step = int(m.group(1))
        if step > best_step:
            best_step, best = step, path
    return best


def sync_resume_checkpoint(source: Path, dest: Path) -> Path:
    """Copy full HF checkpoint (weights + optimizer + scheduler + RNG) to fixed folder."""
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(source, dest)
    (dest / 'RESUME_READY.txt').write_text(
        f'synced_from={source}\nstep={source.name}\n'
        f'at={datetime.now(timezone.utc).isoformat()}\n',
        encoding='utf-8',
    )
    return dest


def resolve_resume_checkpoint(output_dir: Path, resume_dir: Path):
    if (resume_dir / 'trainer_state.json').is_file():
        return resume_dir
    return find_latest_checkpoint(output_dir)


def write_manifest(manifest_path: Path, checkpoint_dir, status, extra=None):
    payload = {
        'status': status,
        'updated_at': datetime.now(timezone.utc).isoformat(),
        'checkpoint_dir': str(checkpoint_dir) if checkpoint_dir else None,
    }
    if checkpoint_dir and (Path(checkpoint_dir) / 'trainer_state.json').is_file():
        state = json.loads((Path(checkpoint_dir) / 'trainer_state.json').read_text())
        payload['global_step'] = state.get('global_step')
        payload['epoch'] = state.get('epoch')
        payload['log_history'] = state.get('log_history', [])[-5:]
    if extra:
        payload.update(extra)
    manifest_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


class DriveResumeCallback(TrainerCallback):
    def __init__(self, output_dir, resume_dir, manifest_path):
        self.output_dir = Path(output_dir)
        self.resume_dir = Path(resume_dir)
        self.manifest_path = Path(manifest_path)

    def on_save(self, args, state, control, **kwargs):
        ckpt = find_latest_checkpoint(self.output_dir)
        if ckpt is None:
            return
        print(f'\n💾 Syncing full checkpoint to Drive: {self.resume_dir}')
        sync_resume_checkpoint(ckpt, self.resume_dir)
        write_manifest(self.manifest_path, ckpt, 'training', {'global_step': state.global_step})
        print(f'   ✓ Resume-ready at step {state.global_step}')

In [ ]:
# @title 5. Load model (4-bit QLoRA)
import torch
# FastLanguageModel imported in cell 3 (unsloth must load first)

assert torch.cuda.is_available(), 'Enable GPU runtime (T4)'
print('GPU:', torch.cuda.get_device_name(0))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# @title 6. Prepare dataset (cached on Drive — skip re-tokenize on resume)
import json
from datasets import Dataset, load_from_disk
from tqdm.auto import tqdm

if TOKENIZED_CACHE.is_dir() and (TOKENIZED_CACHE / 'dataset_info.json').is_file():
    print('Loading tokenized cache from Drive...')
    train_dataset = load_from_disk(str(TOKENIZED_CACHE))
else:
    print('Tokenizing JSONL (one-time, saved to Drive)...')
    rows = []
    with TRAIN_JSONL.open(encoding='utf-8') as f:
        for line in tqdm(f, desc='Reading JSONL'):
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    texts = [format_example(r, tokenizer) for r in tqdm(rows, desc='Formatting')]
    train_dataset = Dataset.from_dict({'text': texts})
    train_dataset.save_to_disk(str(TOKENIZED_CACHE))
    print('Saved tokenized cache:', TOKENIZED_CACHE)

print('Train examples:', len(train_dataset))

In [ ]:
# @title 7. Train — auto-resumes from Drive checkpoint
from trl import SFTConfig, SFTTrainer

resume_ckpt = resolve_resume_checkpoint(OUTPUT_DIR, RESUME_DIR)
if resume_ckpt:
    print(f'↻ RESUMING from: {resume_ckpt}')
    write_manifest(MANIFEST_PATH, resume_ckpt, 'resumed')
else:
    print('▶ Starting fresh training run')
    write_manifest(MANIFEST_PATH, None, 'started')

use_bf16 = torch.cuda.is_bf16_supported()
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    save_strategy='steps',
    warmup_steps=100,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    report_to='none',
    seed=42,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    callbacks=[DriveResumeCallback(OUTPUT_DIR, RESUME_DIR, MANIFEST_PATH)],
)

trainer.train(resume_from_checkpoint=str(resume_ckpt) if resume_ckpt else None)

final_ckpt = find_latest_checkpoint(OUTPUT_DIR)
if final_ckpt:
    sync_resume_checkpoint(final_ckpt, RESUME_DIR)
write_manifest(MANIFEST_PATH, final_ckpt, 'training_complete')
print('\n✓ Training finished')

In [ ]:
# @title 8. Save adapter + export GGUF for Ollama
from unsloth import FastLanguageModel

FINAL_ADAPTER.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
print('Adapter saved:', FINAL_ADAPTER)

# Q4_K_M — good balance for 6GB local VRAM
GGUF_OUTPUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf(
    str(GGUF_OUTPUT / 'jurisguard-phi35-q4'),
    tokenizer,
    quantization_method='q4_k_m',
)
print('GGUF saved under:', GGUF_OUTPUT)
print('\nDownload the .gguf file and import locally:')
print('  ollama create jurisguard -f Modelfile')

## After a crash / disconnect (even days later)

1. Re-open this notebook
2. **Runtime → Run all** (or run cells 1–7)
3. Cell 7 detects `checkpoint_RESUME/` on Drive and continues from the **exact step** (optimizer + gradients restored)

Check progress anytime on Drive:
- `RUN_MANIFEST.json` — last step, status, recent loss
- `checkpoint_RESUME/trainer_state.json` — full trainer state

**Do not** delete `checkpoint_RESUME/` until training is complete.